In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import numpy as np

In [10]:
BASE_URL = "https://api.exchange.coinbase.com"
PRODUCT_ID = "BTC-USD"
GRANULARITY = 3600
MAX_DAYS = 12      

def iso(dt):
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")

def fetch_candles(start, end):
    url = f"{BASE_URL}/products/{PRODUCT_ID}/candles"
    params = {
        "start": iso(start),
        "end": iso(end),
        "granularity": GRANULARITY
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()


start_date = datetime(2016, 1, 31, 23, 0, 0, tzinfo=timezone.utc) # Take from 31 Jan 2016 23:00, so can calculate log return for 1 Feb 2016 00:00
end_date   = datetime(2026, 2, 1, 23, 0, 0, tzinfo=timezone.utc) # To 1 Feb 2026 23:00
all_candles = []
t = start_date

while t < end_date:
    t_next = min(t + timedelta(days=MAX_DAYS), end_date)
    candles = fetch_candles(t, t_next)
    all_candles.extend(candles)
    t = t_next


df = pd.DataFrame(
    all_candles,
    columns=["timestamp", "low", "high", "open", "close", "volume"]
)

df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
df = df.drop_duplicates(subset="datetime")
df = df.sort_values("datetime")
df = df[["datetime", "open", "high", "low", "close", "volume"]]

In [19]:
print(df.tail())
print(df.head())
print(f"Total rows: {len(df)}")
df_hourly = df.copy()
df_hourly = df_hourly.set_index("datetime")

                       datetime      open      high       low     close  \
87810 2026-02-01 19:00:00+00:00  77948.95  78029.06  76892.17  77092.00   
87809 2026-02-01 20:00:00+00:00  77092.00  77663.74  76516.00  77025.05   
87808 2026-02-01 21:00:00+00:00  77021.51  77092.00  76276.58  76461.14   
87807 2026-02-01 22:00:00+00:00  76442.87  77698.00  76372.44  77170.00   
87806 2026-02-01 23:00:00+00:00  77169.98  77547.20  75612.75  76895.53   

            volume  
87810   466.093183  
87809   790.362178  
87808   993.786807  
87807   925.477857  
87806  1285.870772  
                     datetime    open    high     low   close      volume
288 2016-01-31 23:00:00+00:00  374.50  374.79  365.00  367.95  987.279849
287 2016-02-01 00:00:00+00:00  367.89  370.84  367.89  369.29  403.099027
286 2016-02-01 01:00:00+00:00  369.24  369.24  366.26  367.65  461.846216
285 2016-02-01 02:00:00+00:00  367.65  370.83  367.25  370.66  446.643967
284 2016-02-01 03:00:00+00:00  370.66  374.00  370.34

In [20]:
df_hourly

,open,high,low,close,volume
datetime,,,,,
2016-01-31 23:00:00+00:00,374.50,374.79,365.00,367.95,987.279849
2016-02-01 00:00:00+00:00,367.89,370.84,367.89,369.29,403.099027
2016-02-01 01:00:00+00:00,369.24,369.24,366.26,367.65,461.846216
2016-02-01 02:00:00+00:00,367.65,370.83,367.25,370.66,446.643967
2016-02-01 03:00:00+00:00,370.66,374.00,370.34,373.92,439.962414
...,...,...,...,...,...
2026-02-01 19:00:00+00:00,77948.95,78029.06,76892.17,77092.00,466.093183
2026-02-01 20:00:00+00:00,77092.00,77663.74,76516.00,77025.05,790.362178
2026-02-01 21:00:00+00:00,77021.51,77092.00,76276.58,76461.14,993.786807


**Compute hourly log returns**

In [21]:
df_hourly["r_h"] = np.log(df_hourly["close"] / df_hourly["close"].shift(1))
df_hourly = df_hourly.dropna(subset=["r_h"]) # drop 31 Jan 2016

In [22]:
df_hourly

,open,high,low,close,volume,r_h
datetime,,,,,,
2016-02-01 00:00:00+00:00,367.89,370.84,367.89,369.29,403.099027,0.003635
2016-02-01 01:00:00+00:00,369.24,369.24,366.26,367.65,461.846216,-0.004451
2016-02-01 02:00:00+00:00,367.65,370.83,367.25,370.66,446.643967,0.008154
2016-02-01 03:00:00+00:00,370.66,374.00,370.34,373.92,439.962414,0.008757
2016-02-01 04:00:00+00:00,373.88,375.00,372.99,373.50,315.588172,-0.001124
...,...,...,...,...,...,...
2026-02-01 19:00:00+00:00,77948.95,78029.06,76892.17,77092.00,466.093183,-0.011055
2026-02-01 20:00:00+00:00,77092.00,77663.74,76516.00,77025.05,790.362178,-0.000869
2026-02-01 21:00:00+00:00,77021.51,77092.00,76276.58,76461.14,993.786807,-0.007348


**Compute realized variance and realized volatility**
- Variance = sum of hourly squared log returns
- Volatility = sqrt of Variance

In [24]:
daily_realized = (
    df_hourly
    .groupby(df_hourly.index.date)["r_h"]
    .apply(lambda x: np.sum(x**2))
)

daily_realized.index = pd.to_datetime(daily_realized.index)
daily_realized = daily_realized.to_frame(name="realized_variance")
daily_realized["realized_volatility"] = np.sqrt(daily_realized["realized_variance"])

In [25]:
daily_realized

,realized_variance,realized_volatility
2016-02-01,0.000424,0.020584
2016-02-02,0.000075,0.008650
2016-02-03,0.000105,0.010256
2016-02-04,0.000988,0.031438
2016-02-05,0.000221,0.014855
...,...,...
2026-01-28,0.000319,0.017847
2026-01-29,0.001190,0.034490
2026-01-30,0.000855,0.029246
2026-01-31,0.000998,0.031598


In [26]:
daily_realized.isna().sum()

realized_variance      0
realized_volatility    0
dtype: int64

In [27]:
daily_realized.shape

(3654, 2)

In [31]:
daily_realized.to_csv("daily_realized.csv")